# Helper Functions

In [ ]:
import pandas as pd
import os
import json
import random

RANDOM_STATE = 42
random.seed(RANDOM_STATE)

def generate_demographic_prompt(row, excluded_columns): 
    """Build a numbered interview-style prompt string from a row's demographic columns.

    Iterates over the row's columns (excluding those in `excluded_columns`),
    and formats each non-null value as a numbered interviewer–respondent exchange.

    Args:
        row: A pandas Series representing a single survey respondent.
        excluded_columns: List of column names to skip when building the prompt.

    Returns:
        A string of numbered demographic Q&A pairs.
    """
    demographic_questions = [question for question in list(row.index) if question not in excluded_columns]
    # random.shuffle(demographic_questions)
    demographic_prompt = ""
    counter = 1
    for question in demographic_questions:
        if pd.isnull(row[question]) or row[question] == "NA" or row[question] == "N/A":
            continue
        demographic_prompt += f"{counter}) Interviewer: {question} Me: {row[question]} "
        counter += 1
    return demographic_prompt


def save_jsonl(df: pd.DataFrame, path: str, text_column: str = "text"):
    """Save rows as JSONL lines: {"messages": [...]}.
    Expects df[text_column] to be a JSON string or a Python list of message dicts.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            val = row.get(text_column)
            if isinstance(val, str):
                try:
                    messages = json.loads(val)
                except Exception:
                    # keep string as a single assistant message if not JSON
                    messages = [{"role": "assistant", "content": val}]
            elif isinstance(val, list):
                messages = val
            else:
                messages = [{"role": "assistant", "content": str(val)}]

            out = {"messages": messages}
            json.dump(out, f, ensure_ascii=False)
            f.write("\n")

# Duch et al. 2023

In [ ]:
RANDOM_STATE = 42
DATA = "duch_et_al_2023_training_1538"

demographic_questions = [
    "Start Date",
    "What is your current age?",
    "What is your gender?",
    "What is the highest educational qualification you have completed?",
    "Which region do you live in?",
    "Which distric do you live in?",
    "What is the name of the community you live in?",
    "How many people live in your village?",
    "What is the distance in km of the nearest health clinic from where you live?",
    "How many people live in the house together with you (NOT including you) at this moment?",
    "How many children below 18 years old are currently living in your home?",
    "What is your current working situation?",
    "How much (in Ghanaian Cedis) on average does your household spend in a typical week on food?",
    "How much (in Ghanaian Cedis) on average does your household spend in a typical week on non-food items (electricity, water, rent, school fees)?",
    "How would you rate the overall economic or financial condition of your household today?",
    "Do you have a registered mobile number?",
    "How many family members do you have in another village?",
    "How many friends and acquaintances who are not part of your family do you have in another village?",
    "How many individuals can you identify in your social network? Think of friends and relatives that live close to you",
    "How often do you use social media?",
]
    
target_outcome = "Have you received a COVID-19 vaccine, as verified in the records of the Ghanaian District Health Offices?"

system_prompt = """You participate in a healthcare survey in Ghana and have the following profile: 
{demographic_prompt}

During the survey, you received the following information:
{treatment_prompt}

You were told you would be contacted in six weeks to verify your vaccination status. Six weeks have now passed."""

treatment_transcript = {
    "CDC Health": "Health authorities are working hard to distribute the COVID-19 vaccines free for everyone with no strings attached. COVID 19 vaccines are safe and effective. After you have been fully vaccinated you can resume activities that you did prior to the pandemic. Getting the COVID-19 vaccine will help prevent you from getting COVID-19 and reduce your risk of being hospitalized with COVID-19. COVID 19 vaccine help you to protect yourself your environment and your loved ones from COVID-19 exposure.\nWe indicated that we will follow up with you in six weeks. We will contact you in order to verify your vaccination status. If you can provide us with your COVID-19 vaccination carnet at the time, we will upload a copy of the vaccination carnet to our secure server for verification.",
    "Placebo": "The Sun lights up our lives for business for education even for socializing but when the Sun sets many people use candles who are quality battery-operated torches and kerosene lamps as inefficient and expensive ways to create light. What if you can take some Sun with you at night?  You can with portable solar products there are different types, but each portable solar product is made up of three basic parts: a small solar panel, a modern rechargeable battery and an LED bulb. The solar panel catches the light from the Sun and stores this energy in the battery. This can now be used for much needed light when it's dark. Many can even charge phones portable solar products should be reliable affordable and warranted be sure to demand top quality solar products look for these products lighting Africa shining the way.\nWe indicated that we will follow up with you in six weeks. We will contact you in order to verify your vaccination status. If you can provide us with your COVID-19 vaccination carnet at the time, we will upload a copy of the vaccination carnet to our secure server for verification.",
    "Low Cash": "Health authorities are working hard to distribute the COVID-19 vaccines free for everyone with no strings attached. COVID-19 vaccines are safe and effective. After you have been fully vaccinated you can resume activities that you did prior to the pandemic. If you have at least one COVID-19 vaccine shot you will receive 20 Cedi. If you get vaccinated, you will get rewarded.\nWe indicated that we will follow up with you in six weeks. We will contact you in order to verify your vaccination status. If you can provide us with your COVID-19 vaccination carnet at the time, we will upload a copy of the vaccination carnet to our secure server for verification and you will be paid your 20 Cedi via cell phone money payment or by cash if you prefer.",
    "High Cash": "Health authorities are working hard to distribute the COVID-19 vaccines free for everyone with no strings attached. COVID-19 vaccines are safe and effective. After you have been fully vaccinated you can resume activities that you did prior to the pandemic. If you have at least one COVID-19 vaccine shot you will receive 60 Cedi. If you get vaccinated, you will get rewarded.\nWe indicated that we will follow up with you in six weeks. We will contact you in order to verify your vaccination status. If you can provide us with your COVID-19 vaccination carnet at the time, we will upload a copy of the vaccination carnet to our secure server for verification and you will be paid your 60 Cedi via cell phone money payment or by cash if you prefer.",
}


def format_duch_2023_system_prompt(row: pd.Series) -> str:
    """Format the system prompt for a Duch et al. 2023 respondent.

    Populates the system prompt template with the respondent's demographic
    profile and the treatment-specific transcript they received.

    Args:
        row: A pandas Series containing 'demographic_prompt' and 'treatment' columns.

    Returns:
        The fully formatted system prompt string.
    """
    final_system_prompt = system_prompt.format(
        demographic_prompt=row["demographic_prompt"],
        treatment_prompt=treatment_transcript[row["treatment"]],
    )

    return final_system_prompt


def format_duch_2023_user_prompt(row: pd.Series, target_outcome: str) -> str:
    """Build the fine-tuning message sequence for a single respondent.

    Constructs a list of system, user, and assistant messages representing
    a complete conversation turn, then serialises it to a JSON string
    suitable for fine-tuning input.

    Args:
        row: A pandas Series containing 'system_prompt' and the target outcome column.
        target_outcome: The column name of the survey outcome question.

    Returns:
        A JSON string encoding the list of message dicts.
    """
    question_prompt = f"{target_outcome} Please give your response to the question in the structured format below:\n[Yes/No]"
    response = row[target_outcome]

    prompt = [{"role":"system","content":row["system_prompt"]},
              {"role":"user","content":question_prompt},
              {"role":"assistant","content":response}
            ]
        
    # Convert the prompt list to a JSON-formatted string
    prompt_string = json.dumps(prompt)

    return prompt_string

# Load data
data = pd.read_csv(f"data/{DATA}.csv", header=1)
data[target_outcome] = data[target_outcome].replace("NA", None)

# Construct system and question prompts
training_data_finetune = data[demographic_questions + [target_outcome, "treatment"]].copy()
training_data_finetune["demographic_prompt"] = training_data_finetune.apply(generate_demographic_prompt, axis=1, args=([target_outcome, "treatment"],))
training_data_finetune["system_prompt"] = training_data_finetune.apply(format_duch_2023_system_prompt, axis=1)

# Format data for fine-tuning
training_data_finetune["text"] = training_data_finetune.apply(format_duch_2023_user_prompt, axis=1, args=(target_outcome,))
training_data_finetune = training_data_finetune.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Save files in CSV format
data_folder_dir = DATA
if not os.path.exists(f"data/{data_folder_dir}"):
    # Create the folder
    os.makedirs(f"data/{data_folder_dir}")

training_data_finetune[["text"]].to_csv(f"data/{data_folder_dir}/train.csv", index=False)
save_jsonl(training_data_finetune[["text"]], f"data/{data_folder_dir}/{data_folder_dir}_train.jsonl", text_column="text")